# Huấn luyện nhận diện cảm xúc (mặt + tiếng) trên Colab
Chạy lần lượt từng ô. Nhớ bật GPU: **Runtime → Change runtime type → T4 GPU**.

Cần chuẩn bị trên Google Drive: thư mục code `emotion-mm` và file `data.zip` (ở `MyDrive/`).

Toàn bộ log + checkpoint được ghi vào **`MyDrive/emotion-mm/runs/`** (không mất khi phiên ngắt).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT  = "/content/drive/MyDrive/emotion-mm"     # thư mục code (đã upload lên Drive)
DATA_ZIP = "/content/drive/MyDrive/data.zip"       # file data
RUNS     = f"{PROJECT}/runs"                       # log + checkpoint lưu NGAY TRONG thư mục emotion-mm trên Drive
CACHE    = "/content/cache"                        # cache data đã parse: để ở ổ Colab cho nhanh, không tốn dung lượng Drive

import os
os.makedirs(RUNS, exist_ok=True)
print("Log sẽ ghi vào:", RUNS)

In [ ]:
# Chỉ giải nén phần cần cho train (bỏ qua data/raw) -> nhanh hơn, đỡ tốn ổ đĩa
if not os.path.isdir("/content/data/data/splits"):
    !unzip -q "$DATA_ZIP" "data/splits/*" "data/processed/audio/*" -d /content/data
!ls /content/data/data /content/data/data/splits

In [ ]:
%cd $PROJECT
!pip install -q -r requirements.txt
!nvidia-smi | head -15

## 1. Kiểm tra data

In [ ]:
!python scripts/check_data.py --data-root /content/data

## 2. (Khuyên chạy 1 lần) Chạy thử nhanh bằng data giả
Mất khoảng vài phút; kiểm tra GPU + AMP chạy ổn. Nếu ô này lỗi thì sửa trước khi train thật.

In [ ]:
!bash scripts/smoke_test.sh /content/fake_emotion

## 3. Train nhánh MẶT

In [ ]:
!python train.py --config configs/face.yaml --set data_root=/content/data "logging.out_dir=$RUNS" "cache_dir=$CACHE"

## 4. Train nhánh TIẾNG

In [ ]:
!python train.py --config configs/audio.yaml --set data_root=/content/data "logging.out_dir=$RUNS" "cache_dir=$CACHE"

## 4b. (Chỉ khi bị ngắt giữa chừng) Train tiếp từ checkpoint
Chạy lại các ô 0 → 3 (mount Drive, giải nén, cd), rồi điền đường dẫn run bị dở vào `RESUME_RUN`.

In [ ]:
import glob
print("\n".join(sorted(glob.glob(f"{RUNS}/*"))))   # danh sách các run hiện có

RESUME_RUN    = ""                    # vd: f"{RUNS}/face_resnet18_20261115_141530"
RESUME_CONFIG = "configs/face.yaml"   # configs/face.yaml hoặc configs/audio.yaml (đúng nhánh của run)
if RESUME_RUN:
    !python train.py --config "$RESUME_CONFIG" --run-dir "$RESUME_RUN" --resume --set data_root=/content/data "cache_dir=$CACHE"

## 5. Xem log bằng TensorBoard
Có scalar (loss, F1, lr...), norm/độ thay đổi/gradient của TỪNG tham số, và histogram trọng số/gradient.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$RUNS"

## 6. Đồ thị và bảng log
Mặc định lấy run MỚI NHẤT của mỗi nhánh — kiểm tra đường dẫn in ra; nếu đó là run bị dở thì gán tay `FACE_RUN` / `AUDIO_RUN`.

In [ ]:
import glob, pandas as pd
FACE_RUN  = sorted(glob.glob(f"{RUNS}/face_resnet18_*"))[-1]
AUDIO_RUN = sorted(glob.glob(f"{RUNS}/audio_cnn_*"))[-1]
print(FACE_RUN); print(AUDIO_RUN)
for r in (FACE_RUN, AUDIO_RUN):
    if not os.path.exists(f"{r}/results.json"):
        print("CẢNH BÁO: run chưa train xong (thiếu results.json):", r)
for r in (FACE_RUN, AUDIO_RUN):
    !python scripts/plot_logs.py --run-dir "$r"
pd.read_csv(f"{FACE_RUN}/epochs.csv").tail()

In [ ]:
# Các tham số thay đổi mạnh nhất ở epoch cuối (nhánh mặt)
p = pd.read_csv(f"{FACE_RUN}/params.csv")
p[p.epoch == p.epoch.max()].sort_values("update_ratio", ascending=False)[
    ["param","shape","w_norm","update_ratio","grad_norm_mean"]].head(10)

## 7. Fusion (late fusion) mặt + tiếng
Lưu ý: mặt và tiếng không ghép cặp nên cặp được ghép ngẫu nhiên theo nhãn (xem README, mục Hạn chế).
Kết quả ghi vào `emotion-mm/runs/fusion_<thời gian>/`.

In [ ]:
!python fuse_eval.py --face-run "$FACE_RUN" --audio-run "$AUDIO_RUN" --seeds 20

## 8. Đẩy hết log lên Drive (chạy trước khi tắt Colab)
Đảm bảo mọi file vừa ghi đã được đồng bộ lên `MyDrive/emotion-mm/runs/`. Sau ô này Drive bị unmount; muốn dùng tiếp thì chạy lại ô mount.

In [ ]:
!ls -la "$RUNS"
%cd /content
drive.flush_and_unmount()
print("Đã đồng bộ log lên Drive: MyDrive/emotion-mm/runs/")